# JEPA Architecture Explainer (CIFAR-10, **v8**)

A visual walk-through of the **current** JEPA architecture used in `JEPA_CIFAR10.ipynb` **v8**:
i-JEPA-faithful multi-block masking with **disjoint targets**. Every component
is shown end-to-end on a single CIFAR-10 image.

**v8 vs. v7 in one sentence:** the context (now `K_CTX = 100`, ~10 % of the image) and the
**4 contiguous target blocks** (`K_TGT = 50` each, ~5 % each) are **disjoint by construction**
— fixing v7's bug where ~50 % of target coords fell inside the context chunk itself.

**Outline**
1. Config (matches `JEPA_CIFAR10.ipynb` v8)
2. **Data formation (v8)** — image → 40 % pool → 4 target blocks first → 1 context block disjoint
3. Input encoding — RGB + Gaussian Fourier features
4. The two encoders — context (trained, sees K_CTX) vs. target (EMA, sees full 40 % pool)
5. Inside the encoder — `latent_pos` grid + Gaussian attention bias
6. Target side — deterministic Gaussian soft-pool of `g_tgt` over `latent_pos`
7. Predictor side — `proj_g(g) ⊕ (proj_query(γ(c_q)) + mask_token)` → cross-attn
8. End-to-end forward — shapes, loss, per-coord cosine
9. Architecture schematic + tensor-shape table
10. Why v8 should help (compared to v7's plateau)

> Notation: `B` = batch, `N_pix = 32×32 = 1024`, `K_pool = 410` (40 %), `K_HALF = 205` (test-time 20 %),
> `K_CTX = 100` (train context), `K_TGT = 50` per block, `N_PRED = 4` blocks → `N_tgt = 200` total query coords,
> `N_lat = 256`, `D_emb = 512`, `D_pred = 256`, `D_pos = 192`.


## 1. Config — identical to `JEPA_CIFAR10.ipynb` v8

In [ ]:
import os, math, ssl, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
torch.manual_seed(0); np.random.seed(0)

# --- exactly matches JEPA_CIFAR10.ipynb v8 ---
IMAGE_SIZE  = 32
N_PIX       = IMAGE_SIZE * IMAGE_SIZE     # 1024
FRAC_POOL   = 0.40
K_POOL      = int(round(FRAC_POOL * N_PIX))   # 410
K_HALF      = K_POOL // 2                     # 205 (test-time input only)

CHANNELS    = 3
FOURIER_DIM = 96
POS_DIM     = FOURIER_DIM * 2                 # 192
INPUT_DIM   = CHANNELS + POS_DIM              # 195

LATENT_DIMS = (256, 384, 512)
NUM_LATENTS = (256, 256, 256)
EMBED_DIM   = LATENT_DIMS[-1]                 # 512
PRED_DIM    = 256
PRED_LAYERS = 4
PRED_HEADS  = 8
PRED_HEAD_D = 32

# v8: multi-block masking
K_CTX       = 100                # train context block (pool pixels, ~10% of image)
K_TGT       = 50                 # per-target-block (pool pixels, ~5%  of image)
N_PRED      = 4                  # number of target blocks per image
N_TGT       = N_PRED * K_TGT     # = 200

POS_BIAS_SIGMA = 0.5     # encoder Gaussian attention bias σ
POS_POOL_SIGMA = 0.15    # target-side Gaussian soft-pool σ

DEVICE = "cpu"
assert K_CTX + N_PRED*K_TGT <= K_POOL, "infeasible — reduce K_CTX/K_TGT/N_PRED"
print(f"K_POOL={K_POOL}  K_HALF(test)={K_HALF}  K_CTX(train)={K_CTX}")
print(f"N_PRED={N_PRED}  K_TGT={K_TGT}  → N_TGT total = {N_TGT}")
print(f"σ_bias={POS_BIAS_SIGMA}  σ_pool={POS_POOL_SIGMA}")


## 2. Data formation (v8) — pool → 4 target blocks → context disjoint

The v8 sampling matches i-JEPA's `MaskCollator` recipe
(`ijepa/src/masks/multiblock.py`):

```
for each image:
  pool = fixed 40% of pixels (same every epoch)
  # 1. sample N_PRED=4 target blocks (may overlap each other)
  for k in 1..4:
    pick a random pool pixel as anchor_k
    target_block_k = K_TGT=50 nearest pool members of anchor_k
  forbidden = ∪ target_blocks       # union of all target pixels

  # 2. sample CONTEXT block disjoint from targets
  pick a random pool pixel in (pool \ forbidden) as anchor_ctx
  context_block = K_CTX=100 nearest pool members of anchor_ctx within (pool \ forbidden)
```

This guarantees **`context ∩ targets = ∅`** for every image — the predictor
must genuinely *predict* features at unseen spatial locations.


In [ ]:
# Load one CIFAR-10 image
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,)*3, (0.5,)*3)])
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
img, label = cifar[7]
CLASSES = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"Sample label: {CLASSES[label]}  shape={tuple(img.shape)}")

# (y,x) coord grid in [-1, 1]², row-major
ys, xs = torch.meshgrid(
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    indexing='ij',
)
coords_all = torch.stack([ys, xs], dim=-1).view(N_PIX, 2)
pix_all    = img.permute(1, 2, 0).reshape(-1, 3)

# Fixed 40% pool (uses image-specific seed in the real loader)
rng       = np.random.RandomState(12345 + 7)
pool_idx  = rng.permutation(N_PIX)[:K_POOL]
pool_xy   = coords_all[pool_idx]

# --- v8 multi-block sampling ---
# Step 1: sample N_PRED target blocks (may overlap each other)
forbidden = np.zeros(K_POOL, dtype=bool)
target_blocks_local  = []        # local-in-pool indices, one list per block
target_anchors_local = []
for k in range(N_PRED):
    a_local = rng.randint(K_POOL)
    d2 = (pool_xy - pool_xy[a_local]).pow(2).sum(-1).numpy()
    order = np.argsort(d2, kind="stable")
    blk = order[:K_TGT]
    target_blocks_local.append(blk)
    target_anchors_local.append(a_local)
    forbidden[blk] = True

# Step 2: sample CONTEXT block from allowed = pool\forbidden
allowed = np.where(~forbidden)[0]
ctx_anchor_local = allowed[rng.randint(len(allowed))]
d2 = (pool_xy[allowed] - pool_xy[ctx_anchor_local]).pow(2).sum(-1).numpy()
order = np.argsort(d2, kind="stable")
ctx_local = allowed[order[:K_CTX]]

# Resolve to global pixel ids and coords
ctx_idx = pool_idx[ctx_local]
ctx_xy  = coords_all[ctx_idx]
target_blocks_global = [pool_idx[b] for b in target_blocks_local]
target_blocks_xy     = [coords_all[g] for g in target_blocks_global]
tgt_idx = np.concatenate(target_blocks_global)
tgt_xy  = coords_all[tgt_idx]

# Disjointness sanity check
assert len(set(ctx_idx.tolist()) & set(tgt_idx.tolist())) == 0, "ctx ∩ tgt != ∅"
n_uniq_tgt = len(set(tgt_idx.tolist()))
print(f"pool_idx: {pool_idx.shape}   ctx_idx: {ctx_idx.shape}   tgt_idx: {tgt_idx.shape}")
print(f"unique target pixels: {n_uniq_tgt} / {N_TGT}  (overlap among target blocks if < {N_TGT})")
print(f"ctx ∩ tgt = ∅  ✓")


In [ ]:
# 5-panel data visualization (v8): each target block in its own color, disjoint from context.
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']   # 4 distinct colors for 4 target blocks
CTX_COLOR  = '#ef4444'

def img_canvas(highlight_idx=None, alpha=1.0):
    im = ((img.permute(1,2,0) * 0.5) + 0.5).clamp(0,1).numpy()
    if highlight_idx is None:
        return im
    canvas = np.ones_like(im) * 0.85
    flat   = canvas.reshape(-1, 3)
    flat[highlight_idx] = im.reshape(-1, 3)[highlight_idx]
    return canvas

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# (a) original
axes[0].imshow(img_canvas(), extent=(-1, 1, 1, -1))
axes[0].set_title(f"(a) original image\nclass = {CLASSES[label]}", fontsize=10)
axes[0].set_xticks([]); axes[0].set_yticks([])

# (b) 40% pool
axes[1].imshow(img_canvas(pool_idx), extent=(-1, 1, 1, -1))
axes[1].set_title(f"(b) 40% pool\n(K_pool = {K_POOL})", fontsize=10)
axes[1].set_xticks([]); axes[1].set_yticks([])

# (c) 4 target blocks (sampled FIRST in v8)
axes[2].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.32)
for k, xy in enumerate(target_blocks_xy):
    axes[2].scatter(xy[:,1], xy[:,0], s=22, c=TGT_COLORS[k],
                    edgecolors='black', linewidths=0.3, label=f'block {k+1}')
    a = pool_xy[target_anchors_local[k]]
    axes[2].scatter([a[1]], [a[0]], s=120, marker='*', c=TGT_COLORS[k],
                    edgecolors='black', linewidths=1.0, zorder=5)
axes[2].set_xlim(-1.05, 1.05); axes[2].set_ylim(1.05, -1.05)
axes[2].set_xticks([]); axes[2].set_yticks([])
axes[2].set_title(f"(c) {N_PRED} target blocks (sampled FIRST)\n"
                  f"each K_TGT = {K_TGT}  •  ★ = anchor", fontsize=10)
axes[2].legend(loc='lower right', fontsize=6.5, ncol=2)

# (d) context block (disjoint from targets)
axes[3].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.32)
axes[3].scatter(ctx_xy[:,1], ctx_xy[:,0], s=22, c=CTX_COLOR,
                edgecolors='black', linewidths=0.3, label='context')
a = pool_xy[ctx_anchor_local]
axes[3].scatter([a[1]], [a[0]], s=180, marker='*', c='yellow',
                edgecolors='black', linewidths=1.2, zorder=5, label='anchor')
axes[3].set_xlim(-1.05, 1.05); axes[3].set_ylim(1.05, -1.05)
axes[3].set_xticks([]); axes[3].set_yticks([])
axes[3].set_title(f"(d) context block (disjoint from targets)\n"
                  f"K_CTX = {K_CTX}  → encoder input", fontsize=10)
axes[3].legend(loc='lower right', fontsize=7)

# (e) overlay: context (red) + 4 target blocks — visual proof of disjointness
axes[4].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.30)
axes[4].scatter(ctx_xy[:,1], ctx_xy[:,0], s=14, c=CTX_COLOR, alpha=0.85,
                edgecolors='black', linewidths=0.2, label=f'ctx ({K_CTX})')
for k, xy in enumerate(target_blocks_xy):
    axes[4].scatter(xy[:,1], xy[:,0], s=22, c=TGT_COLORS[k],
                    edgecolors='black', linewidths=0.3,
                    label=f'tgt {k+1} ({K_TGT})')
axes[4].set_xlim(-1.05, 1.05); axes[4].set_ylim(1.05, -1.05)
axes[4].set_xticks([]); axes[4].set_yticks([])
axes[4].set_title(f"(e) overlay — disjoint by construction\n"
                  f"ctx ∩ tgt = ∅  (unique tgt = {n_uniq_tgt})", fontsize=10)
axes[4].legend(loc='lower right', fontsize=6, ncol=2)

plt.suptitle("v8 data formation: 4 disjoint contiguous target blocks + 1 context block (also disjoint)",
             y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Input encoding — per-pixel `[RGB ⊕ γ(y,x)]`

The encoder doesn't take the image directly. Each visible pixel becomes a 195-dim
token:

```
input_token[n]  =  [ R, G, B,    γ(y_n, x_n) ]
                    └──3──┘     └────192─────┘
                     pixel       Gaussian Fourier features
```

with `γ(y, x) = [sin(B·c), cos(B·c)]`, `B ∈ ℝ²ˣ⁹⁶` random, `scale=15`.

> v8: the encoder sees **only `K_CTX = 100` pixels** per training step
> (vs. `K_HALF = 205` in v7). It's a sparser context that demands richer features.


In [ ]:
class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

torch.manual_seed(0)
fourier = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0)
ctx_pix = pix_all[ctx_idx]                      # (100, 3)
ctx_pos = fourier(ctx_xy)                       # (100, 192)
ctx_tok = torch.cat([ctx_pix, ctx_pos], dim=-1) # (100, 195)
print(f"ctx_pixels: {tuple(ctx_pix.shape)}     ctx_pos: {tuple(ctx_pos.shape)}     ctx_token: {tuple(ctx_tok.shape)}")

# 3-panel: RGB, γ(y,x) features, concatenated tokens
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(ctx_pix.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f"RGB component  ({K_CTX} × 3)", fontsize=10)
axes[0].set_xlabel("channel"); axes[0].set_ylabel("context pixel #")
axes[0].set_xticks([0,1,2]); axes[0].set_xticklabels(['R','G','B'])

axes[1].imshow(ctx_pos.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title(f"γ(y,x) Fourier features  ({K_CTX} × {POS_DIM})", fontsize=10)
axes[1].set_xlabel("feature dim"); axes[1].set_ylabel("context pixel #")

axes[2].imshow(ctx_tok.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title(f"concatenated input token  ({K_CTX} × {INPUT_DIM})", fontsize=10)
axes[2].set_xlabel("dim (0-2: RGB | 3-194: γ)"); axes[2].set_ylabel("context pixel #")
axes[2].axvline(2.5, color='black', lw=1.5, ls='--')
plt.suptitle("Per-pixel input tokens: `data[n] = [R, G, B, γ(y_n, x_n)]`", fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()


## 4. The two encoders — context (trained) vs. target (EMA)

| Encoder         | Sees what?                              | Weights              | Gradient? |
|-----------------|-----------------------------------------|----------------------|-----------|
| `context_encoder` | **`K_CTX = 100`** context pixels         | trained              | yes       |
| `target_encoder`  | **`K_POOL = 410`** full 40 % pool        | EMA of context (m≈0.999→1.0) | no |

Both have identical OmniField architecture; both expose `latent_pos`.
Target features are computed at the **200 target coordinates** (4 blocks × 50 each)
which lie **outside the context** — so the predictor must genuinely predict, not retrieve.


## 5. Inside the encoder — `latent_pos` + Gaussian attention bias

### 5a. The `latent_pos` grid

256 learnable 2-D anchors on a 16×16 grid in `[-1, 1]²` with tiny jitter.
This is the spatial scaffold the predictor queries against.


In [ ]:
N_lat = NUM_LATENTS[0]
gs = int(math.ceil(math.sqrt(N_lat)))
ys2 = torch.linspace(-1.0, 1.0, gs); xs2 = torch.linspace(-1.0, 1.0, gs)
gy = ys2.unsqueeze(1).expand(gs, gs).contiguous()
gx = xs2.unsqueeze(0).expand(gs, gs).contiguous()
latent_pos = torch.stack([gy, gx], dim=-1).reshape(-1, 2)[:N_lat] + 0.01 * torch.randn(N_lat, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.45)
axes[0].scatter(latent_pos[:,1], latent_pos[:,0], s=22, c='cyan',
                edgecolors='black', linewidths=0.4, label=f'latent_pos ({N_lat})')
axes[0].set_xlim(-1.05,1.05); axes[0].set_ylim(1.05,-1.05); axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title(f"`latent_pos` at init — 16×16 grid in [-1,1]²", fontsize=10)
axes[0].legend(loc='lower right', fontsize=8)

axes[1].scatter(latent_pos[:,1], latent_pos[:,0], s=40, c=np.arange(N_lat),
                cmap='viridis', edgecolors='black', linewidths=0.3)
axes[1].set_xlim(-1.05,1.05); axes[1].set_ylim(1.05,-1.05); axes[1].grid(alpha=0.3)
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].set_title("latent_pos in coord space  (color = latent idx)", fontsize=10)
plt.tight_layout(); plt.show()


### 5b. Gaussian attention bias

Cross-attention logits inside each cascaded block get an additive bias:

$$
\text{sim}(l, n) = \frac{q_l \cdot k_n}{\sqrt{d}} \;-\; \frac{\|\text{latent\_pos}[l] - \text{coord}[n]\|^2}{2\sigma^2},\; \sigma = 0.5
$$

so latent `l` preferentially attends to context pixels near `latent_pos[l]`.
This is what makes `g` position-aware. Below: bias-weighted attention for
3 latents.


In [ ]:
sigma = POS_BIAS_SIGMA
chosen = [0, 128, 245]
d2 = (latent_pos[chosen].unsqueeze(1) - ctx_xy.unsqueeze(0)).pow(2).sum(-1)
bias = -d2 / (2.0 * sigma**2)
attn_no_bias = torch.softmax(torch.zeros_like(bias), dim=-1)
attn_bias    = torch.softmax(bias, dim=-1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col, l in enumerate(chosen):
    a = latent_pos[l]

    ax = axes[0, col]
    ax.imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(ctx_xy[:,1], ctx_xy[:,0], s=22,
                    c=attn_no_bias[col].numpy(), cmap='hot', vmin=0, vmax=attn_bias[col].max()*1.1)
    ax.scatter([a[1]], [a[0]], s=300, marker='*', c='cyan', edgecolors='black', linewidths=1.5, zorder=5)
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"latent {l} — without bias\n(uniform 1/{K_CTX})", fontsize=9)
    plt.colorbar(sc, ax=ax, fraction=0.04)

    ax = axes[1, col]
    ax.imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(ctx_xy[:,1], ctx_xy[:,0], s=22,
                    c=attn_bias[col].numpy(), cmap='hot')
    ax.scatter([a[1]], [a[0]], s=300, marker='*', c='cyan', edgecolors='black', linewidths=1.5, zorder=5)
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"latent {l} — softmax(−d²/2σ²), σ={sigma}", fontsize=9)
    plt.colorbar(sc, ax=ax, fraction=0.04)

plt.suptitle("Gaussian attention bias — each latent sees a soft Gaussian neighborhood around its anchor (★)",
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 6. Target side — deterministic Gaussian soft-pool

For each target coordinate `c_q` (one of the 200 = 4×50 unseen pixels),

$$
w_{q,l} = \mathrm{softmax}_l\!\left(-\frac{\|c_q - \text{latent\_pos}[l]\|^2}{2\sigma_\text{pool}^2}\right),\quad
h_\text{tgt}[q] = \sum_l w_{q,l}\, g_\text{tgt}[l]
$$

with `σ_pool = 0.15`. **No learnable parameters on the target side** — `h_tgt`
is a fixed function of `g_tgt` and the query coord.

> v8 note: the target coords `c_q` lie **outside** the context, but the target
> *encoder* sees the full 40 % pool — so `g_tgt` carries information about those
> pixels. The predictor (working only from the 100 context pixels) must learn
> to extract that.


In [ ]:
sigma_pool = POS_POOL_SIGMA
# Pick 3 query coords from different target blocks
example_q = torch.stack([target_blocks_xy[0][5], target_blocks_xy[1][10], target_blocks_xy[2][20]])

d2 = (example_q.unsqueeze(1) - latent_pos.unsqueeze(0)).pow(2).sum(-1)   # (3, 256)
w  = torch.softmax(-d2 / (2.0 * sigma_pool**2), dim=-1)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col in range(3):
    ax = axes[0, col]
    ax.imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.3)
    sc = ax.scatter(latent_pos[:,1], latent_pos[:,0], s=30,
                    c=w[col].numpy(), cmap='viridis', edgecolors='black', linewidths=0.2)
    q = example_q[col]
    ax.scatter([q[1]], [q[0]], s=400, marker='X', c='red', edgecolors='black',
               linewidths=1.5, zorder=5, label='target coord')
    ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"target coord (block {col+1}, y={q[0]:.2f}, x={q[1]:.2f})\n"
                 f"weight over 256 latents (σ={sigma_pool})", fontsize=9)
    ax.legend(loc='lower right', fontsize=7)
    plt.colorbar(sc, ax=ax, fraction=0.04)

    ax = axes[1, col]
    ws = sorted(w[col].numpy(), reverse=True)
    ax.bar(range(20), ws[:20], color='steelblue', edgecolor='black')
    eff_k = 1.0 / (w[col]**2).sum().item()
    ax.set_xlabel("latent rank (top 20)"); ax.set_ylabel("weight")
    ax.set_title(f"top-20 weights  |  effective #latents = {eff_k:.1f}", fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle("Soft-pool: each target coord ≈ weighted avg of nearby latents (σ_pool=0.15)",
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 7. Predictor side — `CoordReadout`

A small 4-layer transformer (dim 256, 8 heads):

- **Context tokens**: `proj_g(g_ctx)` ∈ ℝ²⁵⁶×²⁵⁶ — pure content
- **Target tokens**: `proj_query(γ(c_q)) + mask_token` ∈ ℝ²⁰⁰×²⁵⁶ — one per target coord

Concatenated, run through self-attention + FFN, then the **target-token slice**
is projected back to `D_emb = 512` and matched to `h_tgt` via smooth-L1.

```
  ┌─ proj_g(g_ctx) ─────────────────┐    (B, 256, 256)
  │                                 │
  │     [   self-attn × 4   ]       │  →  take last N_tgt = 200 tokens
  │                                 │
  └─ proj_query(γ(c_q)) + mask ─────┘    (B, 200, 256)
```


## 8. Full model definitions (matches v8 training)

In [ ]:
def get_sinusoidal_embeddings(n, d):
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None:
            kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)

class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return x * F.gelu(gates)

class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult * 2), GEGLU(),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x): return self.net(x)

class Attention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None, attn_bias=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        if attn_bias is not None:
            B = x.shape[0]; H = sim.shape[0] // B
            ab = attn_bias.unsqueeze(1).expand(-1, H, -1, -1).reshape(B*H, attn_bias.shape[1], attn_bias.shape[2])
            sim = sim + ab
        attn = sim.softmax(dim=-1)
        out  = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))

class CascadedBlock(nn.Module):
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None, pos_bias_sigma=0.5):
        super().__init__()
        self.latents = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads, dim_head=cross_dim_head),
                                  context_dim=input_dim)
        self.self_attn  = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff = PreNorm(dim, FeedForward(dim))
        self.pos_bias_sigma = pos_bias_sigma
    def forward(self, context, residual=None, latent_pos=None, input_coords=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        attn_bias = None
        if latent_pos is not None and input_coords is not None:
            d2 = (latent_pos.unsqueeze(0).unsqueeze(2) - input_coords.unsqueeze(1)).pow(2).sum(-1)
            attn_bias = -d2 / (2.0 * self.pos_bias_sigma**2)
        x = self.cross_attn(x, context=context, attn_bias=attn_bias) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x

class JEPAEncoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128, self_heads=8, self_dim_head=128,
                 num_trunk_layers=4, pos_bias_sigma=POS_BIAS_SIGMA):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev,
                              pos_bias_sigma=pos_bias_sigma))
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([
                PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                PreNorm(final, FeedForward(final)),
            ]) for _ in range(num_trunk_layers)
        ])
        N_lat = num_latents[0]
        gs = max(2, int(math.ceil(math.sqrt(N_lat))))
        ys = torch.linspace(-1.0, 1.0, gs); xs = torch.linspace(-1.0, 1.0, gs)
        gy = ys.unsqueeze(1).expand(gs, gs).contiguous()
        gx = xs.unsqueeze(0).expand(gs, gs).contiguous()
        grid = torch.stack([gy, gx], dim=-1).reshape(-1, 2)[:N_lat]
        grid = grid + 0.01 * torch.randn_like(grid)
        self.latent_pos = nn.Parameter(grid, requires_grad=True)
    def forward(self, data, input_coords):
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual,
                             latent_pos=self.latent_pos, input_coords=input_coords)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        return residual, self.latent_pos.unsqueeze(0).expand(data.size(0), -1, -1)

class CoordReadout(nn.Module):
    def __init__(self, latent_dim=EMBED_DIM, predictor_dim=PRED_DIM,
                 query_in_dim=POS_DIM, num_layers=PRED_LAYERS,
                 heads=PRED_HEADS, dim_head=PRED_HEAD_D):
        super().__init__()
        self.proj_g     = nn.Linear(latent_dim, predictor_dim)
        self.proj_query = nn.Linear(query_in_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            nn.ModuleList([
                PreNorm(predictor_dim, Attention(predictor_dim, heads=heads, dim_head=dim_head)),
                PreNorm(predictor_dim, FeedForward(predictor_dim)),
            ]) for _ in range(num_layers)
        ])
        self.norm     = nn.LayerNorm(predictor_dim)
        self.proj_out = nn.Linear(predictor_dim, latent_dim)
    def forward(self, g, query_coords_enc):
        B, N_lat, _ = g.shape
        ctx_tok = self.proj_g(g)
        tgt_tok = self.proj_query(query_coords_enc) + self.mask_token
        x = torch.cat([ctx_tok, tgt_tok], dim=1)
        for sa, ff in self.blocks:
            x = sa(x) + x
            x = ff(x) + x
        x = self.norm(x)
        return self.proj_out(x[:, N_lat:])

def soft_pool_targets(g, latent_pos_p, query_coord, sigma):
    d2 = (query_coord.unsqueeze(2) - latent_pos_p.unsqueeze(0).unsqueeze(0)).pow(2).sum(-1)
    w  = F.softmax(-d2 / (2.0 * sigma**2), dim=-1)
    return torch.einsum('b q l, b l d -> b q d', w, g)

torch.manual_seed(0)
fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0)
context_encoder = JEPAEncoder()
target_encoder  = copy.deepcopy(context_encoder)
predictor       = CoordReadout()
for p in target_encoder.parameters(): p.requires_grad_(False)

n_enc  = sum(p.numel() for p in context_encoder.parameters())
n_pred = sum(p.numel() for p in predictor.parameters())
print(f"context_encoder: {n_enc/1e6:.2f}M params")
print(f"predictor      : {n_pred/1e6:.2f}M params")
print(f"target_encoder : {n_enc/1e6:.2f}M params  (frozen, EMA-updated)")


## 9. End-to-end forward — v8 shapes (`K_CTX = 100`, `N_TGT = 200`)

In [ ]:
def encode_input(pixels, coords): return torch.cat([pixels, fourier_encoder(coords)], dim=-1)
def encode_query(coords):              return fourier_encoder(coords)

# Batch dim of 1
ctx_p_b  = pix_all[ctx_idx].unsqueeze(0)
ctx_c_b  = ctx_xy.unsqueeze(0)
pool_p_b = pix_all[pool_idx].unsqueeze(0)
pool_c_b = coords_all[pool_idx].unsqueeze(0)
tgt_c_b  = tgt_xy.unsqueeze(0)

ctx_data  = encode_input(ctx_p_b,  ctx_c_b)
pool_data = encode_input(pool_p_b, pool_c_b)
tgt_q_enc = encode_query(tgt_c_b)

print(f"shapes entering the encoders:")
print(f"  ctx_data   : {tuple(ctx_data.shape)}    ← context_encoder input  (K_CTX={K_CTX})")
print(f"  pool_data  : {tuple(pool_data.shape)}    ← target_encoder input   (K_POOL={K_POOL})")
print(f"  tgt_q_enc  : {tuple(tgt_q_enc.shape)}    ← query coords for predictor  (N_TGT={N_TGT})")

with torch.no_grad():
    g_tgt, _   = target_encoder(pool_data, pool_c_b)
    h_tgt_raw  = soft_pool_targets(g_tgt, target_encoder.latent_pos, tgt_c_b, POS_POOL_SIGMA)
    h_tgt      = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))
    g_ctx, _   = context_encoder(ctx_data, ctx_c_b)
    h_pred     = predictor(g_ctx, tgt_q_enc)
    loss       = F.smooth_l1_loss(h_pred, h_tgt)
    cos        = F.cosine_similarity(h_pred, h_tgt, dim=-1)

print(f"\nintermediate features:")
print(f"  g_tgt      : {tuple(g_tgt.shape)}      ← target encoder latent set")
print(f"  h_tgt_raw  : {tuple(h_tgt_raw.shape)}    ← soft-pool target features (pre-LayerNorm)")
print(f"  g_ctx      : {tuple(g_ctx.shape)}      ← context encoder latent set")
print(f"  h_pred     : {tuple(h_pred.shape)}    ← predictor output (matches h_tgt shape)")
print(f"\nloss = {loss.item():.4f}      cos(h_pred, h_tgt) = {cos.mean():.3f} ± {cos.std():.3f}")


In [ ]:
# 6-panel viz: per-coord loss / per-coord cosine / cosine histogram / h_pred / h_tgt / residual
# v8 twist: color each target coord by which target block it belongs to
fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 3, hspace=0.32, wspace=0.28)

# Reconstruct per-target-block coord ranges
block_idx_per_coord = np.repeat(np.arange(N_PRED), K_TGT)   # shape (200,)
per_loss = F.smooth_l1_loss(h_pred[0], h_tgt[0], reduction='none').mean(-1).numpy()
per_cos  = cos[0].numpy()

ax = fig.add_subplot(gs[0, 0])
ax.imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.35)
sc = ax.scatter(tgt_xy[:,1], tgt_xy[:,0], s=40, c=per_loss, cmap='Reds',
                edgecolors='black', linewidths=0.3)
# outline each target coord by block color
for k in range(N_PRED):
    sel = (block_idx_per_coord == k)
    ax.scatter(tgt_xy[sel,1], tgt_xy[sel,0], s=70, facecolors='none',
               edgecolors=TGT_COLORS[k], linewidths=1.2)
ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"(a) per-coord smooth-L1 loss\ntotal = {loss.item():.4f}  •  ring color = block id", fontsize=10)
plt.colorbar(sc, ax=ax, fraction=0.04)

ax = fig.add_subplot(gs[0, 1])
ax.imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.35)
sc = ax.scatter(tgt_xy[:,1], tgt_xy[:,0], s=40, c=per_cos, cmap='RdBu_r',
                vmin=-1, vmax=1, edgecolors='black', linewidths=0.3)
ax.set_xlim(-1.05,1.05); ax.set_ylim(1.05,-1.05); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"(b) per-coord cos(h_pred, h_tgt)\nmean = {cos.mean():.3f}", fontsize=10)
plt.colorbar(sc, ax=ax, fraction=0.04)

ax = fig.add_subplot(gs[0, 2])
# stacked histogram by block
bins = np.linspace(-1, 1, 21)
for k in range(N_PRED):
    sel = (block_idx_per_coord == k)
    ax.hist(per_cos[sel], bins=bins, alpha=0.65, label=f'block {k+1}', color=TGT_COLORS[k], edgecolor='black')
ax.axvline(cos.mean().item(), color='red', ls='--', label=f'mean={cos.mean():.3f}')
ax.set_xlabel("cosine similarity"); ax.set_ylabel("count")
ax.set_xlim(-1, 1); ax.set_title("(c) cosine dist. — per block (untrained init)", fontsize=10)
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = fig.add_subplot(gs[1, 0])
ax.imshow(h_pred[0, :, :64].numpy(), aspect='auto', cmap='RdBu_r')
ax.set_title(f"(d) h_pred[:, :64]   shape = ({N_TGT}, 512)", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")
for k in range(1, N_PRED):
    ax.axhline(k*K_TGT - 0.5, color='black', lw=0.8)

ax = fig.add_subplot(gs[1, 1])
ax.imshow(h_tgt[0, :, :64].numpy(), aspect='auto', cmap='RdBu_r')
ax.set_title(f"(e) h_tgt[:, :64]   shape = ({N_TGT}, 512)", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")
for k in range(1, N_PRED):
    ax.axhline(k*K_TGT - 0.5, color='black', lw=0.8)

ax = fig.add_subplot(gs[1, 2])
diff = (h_pred - h_tgt)[0, :, :64].numpy()
m = max(abs(diff.min()), abs(diff.max()))
ax.imshow(diff, aspect='auto', cmap='RdBu_r', vmin=-m, vmax=m)
ax.set_title("(f) residual = h_pred − h_tgt", fontsize=10)
ax.set_xlabel("embed dim (0-63 shown)"); ax.set_ylabel("target coord")
for k in range(1, N_PRED):
    ax.axhline(k*K_TGT - 0.5, color='black', lw=0.8)

plt.suptitle("Forward pass on one CIFAR-10 image (v8, untrained init): "
             "block id annotated for per-coord views", fontweight='bold', y=0.995)
plt.show()


## 10. Full architecture schematic (v8)

```
                              ┌──────────────────────────────────────────────┐
                              │           CIFAR-10 image (32×32×3)           │
                              └──────────────────────────────────────────────┘
                                       │            │            │
                  fixed 40% pool ──────┘            │            │
                       (410 pixels)                 │            │
                              │                     │            │
   ─── v8 sampling ─────┐     │                     │            │
   1) sample 4 target  │     │                     │            │
      blocks (50 each)  │     │                     │            │
   2) sample 1 context  │     │                     │            │
      block (100), disj │     │                     │            │
   ─────────────────────┘     │                     │            │
                              │                     │            │
                  ┌───────────┴───────────┐         │            │
                  │ context block (100)   │         │            │
                  │ 4 target blocks (200) │         │            │
                  │   disjoint ✓          │         │            │
                  └─────┬─────────────┬───┘         │            │
                        │             │             │            │
   per-pixel tokens     │             │             │            │
   [R,G,B, γ(y,x)] ─────┼─────────────┼─────────────┘            │
        195-d           │             │                          │
                        │             │             pool tokens (410 × 195)
                        ▼             ▼                          ▼
              ┌──────────────────┐  ┌─────────────────┐  ┌──────────────────┐
              │ context_encoder  │  │ target_encoder  │  │  pool input data │
              │  (trained,       │  │  (EMA of ctx,   │  └──────────────────┘
              │   3-stage ICMR + │  │   frozen)       │           │
              │   4-layer trunk) │  │   ICMR+trunk    │◄──────────┘
              │   + latent_pos   │  │   + latent_pos  │
              └────────┬─────────┘  └────────┬────────┘
                       │ g_ctx               │ g_tgt
                       │ (1, 256, 512)       │ (1, 256, 512)
                       │                     │
                       │                     │  deterministic Gaussian
                       │                     │  soft-pool (σ=0.15)
                       │                     │  over latent_pos
                       │                     ▼
                       │           ┌─────────────────────┐
                       │           │     h_tgt_raw       │
                       │           │     (1, 200, 512)   │
                       │           │   LayerNorm →h_tgt  │
                       │           └─────────────────────┘
                       │                     │
                       ▼                     │
              ┌────────────────────┐         │
              │     predictor      │         │
              │  proj_g(g_ctx)     │         │
              │  ⊕                 │         │
              │  proj_query(γ(c_q))│         │
              │   + mask_token     │         │
              │  → 4-layer SA      │         │
              │  → proj_out        │         │
              └─────────┬──────────┘         │
                        │ h_pred             │
                        │ (1, 200, 512)      │
                        │                    │
                        └─────►  smooth_L1(h_pred, h_tgt)  ◄──────┘
                                       │
                                       └─► back-prop to context_encoder + predictor only
                                           target_encoder updated via EMA from context
```

### Tensor-shape table (v8)

| Stage                 | Tensor       | Shape (single image)  | Change vs. v7                          |
|-----------------------|--------------|-----------------------|----------------------------------------|
| pool pixels           | `pool_p`     | `(410, 3)`            | same                                   |
| pool coords           | `pool_c`     | `(410, 2)`            | same                                   |
| context pixels        | `ctx_p`      | `(100, 3)`            | **was (205, 3)** — context shrunk      |
| context coords        | `ctx_c`      | `(100, 2)`            | **was (205, 2)**                       |
| target coords         | `tgt_c`      | `(200, 2)`            | **was (64, 2)** — 4 blocks of 50       |
| input tokens          | `ctx_data`   | `(100, 195)`          | **was (205, 195)**                     |
| target enc input      | `pool_data`  | `(410, 195)`          | same                                   |
| latent set (ctx)      | `g_ctx`      | `(256, 512)`          | same                                   |
| latent set (tgt)      | `g_tgt`      | `(256, 512)`          | same                                   |
| target features       | `h_tgt`      | `(200, 512)`          | **was (64, 512)**                      |
| predictor input       | `tgt_q_enc`  | `(200, 192)`          | **was (64, 192)**                      |
| prediction            | `h_pred`     | `(200, 512)`          | **was (64, 512)**                      |
| loss                  | scalar       | `()`                  | same                                   |

### Loss (unchanged from v7)

$$
\mathcal{L} \;=\; \frac{1}{N_\text{tgt} \cdot D_\text{emb}} \sum_{q,d} \mathrm{smooth\_L1}\Big(h_\text{pred}[q,d],\; h_\text{tgt}[q,d]\Big)
$$


## 11. Why v8 should break v7's plateau

### v7's symptoms
- `Lj` plateau ≈ 0.10–0.19 after a few epochs
- `cos(h_pred, h_tgt) ≈ 0.85`, never climbs
- Probe accuracy plateau 26–29 % at epochs 20–60

### Root cause (v7)
Target coords were sampled uniformly from the **entire 40 % pool**, but the
context was the 205 nearest pool members of a random anchor. So roughly
**half the target coords sat *inside* the context chunk itself** — meaning
the predictor's job was largely *retrieval of a pixel the encoder just saw*,
not real prediction. Trivially-easy task ⇒ trivially-easy plateau.

### What v8 changes
- **Targets sampled first** (4 contiguous blocks of 50). Build `forbidden = ∪ blocks`.
- **Context sampled in pool \ forbidden**. Disjointness is invariant by construction.
- Predictor now has to extract features for **200 genuinely-unseen pixels** per image,
  from the **100 visible pixels** it does have access to.
- The target encoder still sees the full 40 % pool, so `h_tgt` carries the
  ground-truth signal — the predictor's job is to recover it from the context only.

### What carries over (preserved innovations)
1. Learnable `latent_pos` 16×16 grid
2. Gaussian attention bias in encoder cross-attn
3. Deterministic Gaussian soft-pool target (no learnable target readout)
4. No `proj_pos` on context tokens (v5)
5. DINO-style EMA centering of target
6. EMA target encoder (m: 0.999→1.0)
7. No aux variance loss (v6)
8. No predictor warmup (v6)
9. Embedded 3-epoch probe every 20 epochs (v7) — now uses K_HALF=205 loader for consistency

### Sampling invariants (verified standalone, 1000 samples)
- 0 `ctx ∩ tgt` violations
- Context size always exactly `K_CTX = 100`
- Allowed-set `(pool \ targets)` always ≥ `K_CTX` (mean ≈ 245, min 210)
- Unique target pixels per sample mean ≈ 163/200 (some inter-target overlap, matches i-JEPA)
